# Etapa 3.1 - MapReduce con Spark RDDs
**TP IBD - Procesamiento con Spark (PySpark)**

Dominio: comercio minorista de suplementos deportivos.

En esta notebook se implementan **tres procesamientos MapReduce** usando la API de **RDDs** de PySpark, sobre los datos generados en la Etapa 1. Para cada consulta se documenta:
1. La pregunta de negocio.
2. La **Fase de Map**: transformación aplicada y estructura `(clave, valor)` generada.
3. La **Fase de Reduce**: operación de agregación y sobre qué claves.
4. El resultado final, impreso y comentado.

> **Nota sobre los datos:** se asume que las tablas de la Etapa 1 ya fueron exportadas a CSV (con encabezado) en la carpeta `data/`. El paso de exportación queda fuera de esta notebook.

> **Nota sobre lazy evaluation:** `map`, `filter` y `reduceByKey` son **transformaciones perezosas (lazy)**: construyen el DAG pero no computan nada. Recién al invocar una **acción** (`collect`, `take`, `takeOrdered`) Spark ejecuta el plan, aplicando combinadores locales por partición antes del *shuffle*.

## Configuración del entorno Spark

In [ ]:
from pyspark import SparkContext, SparkConf

# Contexto local usando todos los núcleos disponibles ('local[*]').
conf = SparkConf().setAppName("TP_IBD_MapReduce").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("WARN")
print("SparkContext activo:", sc.version)

## Carga de los CSV y utilidades de parseo

Leemos cada CSV como RDD de texto con `textFile`. La primera línea es el encabezado: la usamos para construir un mapa `nombre_columna -> índice`, de modo que el acceso a las columnas sea robusto al orden en que fueron exportadas.

Los campos del dominio (fechas, horas, números, métodos de pago, estados) no contienen comas, por lo que un `split(',')` simple es seguro para este dataset.

In [ ]:
DATA_DIR = "data"  # carpeta donde residen los CSV exportados de la Etapa 1


def cargar_csv(nombre_archivo):
    """Lee un CSV con encabezado y devuelve (rdd_de_filas, dict_columna->indice).

    - rdd_de_filas: RDD donde cada elemento es una lista de strings (una fila, sin el header).
    - col: diccionario {nombre_columna: posicion} para acceder por nombre.
    """
    rdd_texto = sc.textFile(f"{DATA_DIR}/{nombre_archivo}")
    header = rdd_texto.first()
    col = {nombre: i for i, nombre in enumerate(header.split(","))}
    # Filtramos la línea de encabezado y separamos cada fila en campos
    rdd_filas = (
        rdd_texto
        .filter(lambda linea: linea != header)
        .map(lambda linea: linea.split(","))
    )
    return rdd_filas, col


# RDDs base reutilizados por las tres consultas
ventas_rdd, V = cargar_csv("ventas.csv")
detalle_ventas_rdd, DV = cargar_csv("detalle_ventas.csv")

# Cacheamos porque se reutilizan en varias consultas (evita re-leer el CSV)
ventas_rdd.cache()
detalle_ventas_rdd.cache()

print("Columnas VENTAS:", V)
print("Columnas DETALLE_VENTAS:", DV)

---
## Consulta 1 — Facturación total y ticket promedio por sucursal

**1. Pregunta de negocio:** ¿Cuánto facturó cada punto de venta en el período y cuál es su *ticket promedio* (monto medio por venta)? Permite comparar el rendimiento de las sucursales y detectar dónde el volumen es alto pero el ticket bajo (o viceversa).

**2. Fase de Map:** por cada venta se emite la **clave `puntos_de_venta_id`** con el **valor tupla `(precio_total, 1)`**. Se arrastra el contador `1` para poder calcular el promedio en el reduce.

$$\text{venta} \rightarrow (\text{puntos\_de\_venta\_id},\ (\text{precio\_total},\ 1))$$

**3. Fase de Reduce:** `reduceByKey` sobre `puntos_de_venta_id`, sumando las tuplas componente a componente → `(suma_facturacion, cantidad_ventas)`. La operación es asociativa y conmutativa, lo que habilita el *map-side combine* antes del shuffle. Un `mapValues` final deriva el ticket promedio.

In [ ]:
# --- FASE MAP ---
# (puntos_de_venta_id, (precio_total, 1))
map_c1 = ventas_rdd.map(
    lambda f: (f[V["puntos_de_venta_id"]], (float(f[V["precio_total"]]), 1))
)

# --- FASE REDUCE ---
# Suma componente a componente: (sum_monto, sum_conteo) por sucursal
reduce_c1 = map_c1.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Derivar (facturacion_total, cantidad_ventas, ticket_promedio)
resultado_c1 = reduce_c1.mapValues(
    lambda v: (round(v[0], 2), v[1], round(v[0] / v[1], 2))
)

# --- ACCIÓN: dispara el cómputo ---
print("Sucursal | Facturación total | Cant. ventas | Ticket promedio")
print("-" * 65)
for pv_id, (fact, cant, ticket) in sorted(resultado_c1.collect(), key=lambda x: -x[1][0]):
    print(f"  {pv_id:>3}    | {fact:>16,.2f} | {cant:>11} | {ticket:>14,.2f}")

**Comentario del resultado:** se obtiene una fila por sucursal con su facturación acumulada, la cantidad de operaciones y el ticket promedio. Como las ventas se reparten de forma aproximadamente uniforme entre las 4 sucursales, se esperan facturaciones del mismo orden de magnitud; las diferencias de *ticket promedio* reflejan la mezcla de productos/combos vendidos en cada una.

---
## Consulta 2 — Top 10 productos por profit (mayor valor)

**1. Pregunta de negocio:** ¿Cuáles son los 10 productos que más ganancia (`profit`) generaron en total? Identifica los productos *estrella* sobre los que conviene asegurar stock y enfocar la estrategia comercial.

**2. Fase de Map:** se **filtran** las líneas de producto (descartando las de combo, que tienen `product_id` vacío) y por cada una se emite la **clave `product_id`** con el **valor `profit`**.

$$\text{detalle\_venta (product\_id no nulo)} \rightarrow (\text{product\_id},\ \text{profit})$$

**3. Fase de Reduce:** `reduceByKey` sobre `product_id` sumando el profit → profit total por producto. El *Top 10* se obtiene con la acción `takeOrdered`, que ordena de forma distribuida (parcial por partición + combinación) sin traer todo el RDD al driver.

In [ ]:
# --- FASE MAP (con filtro previo) ---
# Descartamos las líneas de combo: product_id vacío ('' o 'NULL' según export)
def es_linea_de_producto(f):
    pid = f[DV["product_id"]].strip()
    return pid != "" and pid.upper() != "NULL"


# (product_id, profit)
map_c2 = (
    detalle_ventas_rdd
    .filter(es_linea_de_producto)
    .map(lambda f: (f[DV["product_id"]], float(f[DV["profit"]])))
)

# --- FASE REDUCE ---
# Profit total acumulado por producto
reduce_c2 = map_c2.reduceByKey(lambda a, b: a + b)

# --- ACCIÓN: Top 10 por profit (distribuida) ---
top10_c2 = reduce_c2.takeOrdered(10, key=lambda x: -x[1])

print("Top 10 productos por profit total")
print("Pos | product_id | Profit total")
print("-" * 40)
for pos, (pid, profit) in enumerate(top10_c2, start=1):
    print(f" {pos:>2} | {pid:>10} | {profit:>14,.2f}")

**Comentario del resultado:** la lista muestra los 10 `product_id` con mayor ganancia acumulada, de mayor a menor. Suelen encabezarla productos de categorías de alto margen y precio (p. ej. *Isolate Whey Protein* o *Pre-Entrenamiento C4*), combinando volumen de ventas y margen unitario. Para reportarlo con nombres se podría unir luego contra `productos.csv` por `product_id`.

---
## Consulta 3 — Distribución de ventas e ingresos por método de pago

**1. Pregunta de negocio:** ¿Cómo se reparten las ventas entre los distintos métodos de pago, en cantidad de operaciones y en monto facturado? Útil para negociar comisiones con tarjetas/billeteras y entender el comportamiento de pago de los clientes.

**2. Fase de Map:** por cada venta se emite la **clave `metodo_pago`** con el **valor tupla `(1, precio_total)`** (frecuencia + monto en una sola pasada).

$$\text{venta} \rightarrow (\text{metodo\_pago},\ (1,\ \text{precio\_total}))$$

**3. Fase de Reduce:** `reduceByKey` sobre `metodo_pago`, sumando ambos componentes → `(cantidad_ventas, monto_total)` por método. Como hay solo 6 categorías el shuffle es mínimo. Un paso final calcula el porcentaje sobre el total.

In [ ]:
# --- FASE MAP ---
# (metodo_pago, (1, precio_total))
map_c3 = ventas_rdd.map(
    lambda f: (f[V["metodo_pago"]], (1, float(f[V["precio_total"]])))
)

# --- FASE REDUCE ---
# (cantidad_ventas, monto_total) por método de pago
reduce_c3 = map_c3.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# --- ACCIÓN ---
resultado_c3 = reduce_c3.collect()

# Total global para calcular porcentajes (sobre la cantidad de ventas)
total_ventas = sum(cant for _, (cant, _) in resultado_c3)

print("Método de pago    | Cant. ventas |   % | Monto total")
print("-" * 60)
for metodo, (cant, monto) in sorted(resultado_c3, key=lambda x: -x[1][0]):
    pct = 100.0 * cant / total_ventas
    print(f"{metodo:<17} | {cant:>11} | {pct:>4.1f} | {monto:>14,.2f}")

**Comentario del resultado:** se obtiene la distribución de las ~5200 ventas entre los 6 métodos de pago. Como el generador asigna el método de forma uniforme, se espera un reparto cercano a ~16.7% por método tanto en cantidad como en monto; en un escenario productivo real esta consulta revelaría sesgos (p. ej. predominio de MercadoPago o tarjetas) relevantes para la negociación de comisiones.

---
## Cierre

In [ ]:
# Liberamos los RDD cacheados y detenemos el contexto
ventas_rdd.unpersist()
detalle_ventas_rdd.unpersist()
sc.stop()
print("SparkContext detenido.")